# CAR-T Target Expression Across Immune Ecotypes
Analyzes B4GALNT1 (GD2), ST8SIA1 (GD3), IL13RA2, ERBB2 (HER2), CD276 (B7-H3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations

tpm = pd.read_csv('tpm_for_cibersortx.tsv', sep='\t', index_col=0)
eco = pd.read_csv('ecotype_assignment_k3_annotated.tsv')

targets = {'B4GALNT1': 'B4GALNT1', 'ST8SIA1': 'ST8SIA1', 'IL13RA2': 'IL13RA2', 'ERBB2': 'ERBB2', 'CD276': 'CD276'}
samples = eco['Kids_First_Biospecimen_ID'].tolist()
gene_symbols = list(targets.values())
expr = tpm.loc[tpm.index.isin(gene_symbols), [s for s in samples if s in tpm.columns]].T
expr_log = np.log2(expr + 1).reset_index()
expr_log.columns = ['Kids_First_Biospecimen_ID'] + gene_symbols
df = expr_log.merge(eco[['Kids_First_Biospecimen_ID', 'ecotype']], on='Kids_First_Biospecimen_ID')
print(f'Merged: {len(df)} samples')
df['ecotype'].value_counts()

In [ ]:
ecotype_order = ['Inflamed', 'Intermediate', 'Immune-desert']
pairs = list(combinations(ecotype_order, 2))

for gene in gene_symbols:
    groups = [df.loc[df['ecotype']==e, gene].values for e in ecotype_order]
    H, p = kruskal(*groups)
    medians = {e: 2**df.loc[df['ecotype']==e, gene].median() - 1 for e in ecotype_order}
    print(f'\n{gene}: H={H:.1f}, p={p:.2e}')
    print(f'  Median TPM: Infl={medians["Inflamed"]:.1f}, Inter={medians["Intermediate"]:.1f}, Desert={medians["Immune-desert"]:.1f}')
    for e1, e2 in pairs:
        _, p_mwu = mannwhitneyu(df.loc[df['ecotype']==e1, gene].values, df.loc[df['ecotype']==e2, gene].values)
        p_adj = min(p_mwu * 3, 1)
        sig = '***' if p_adj < 0.001 else '**' if p_adj < 0.01 else '*' if p_adj < 0.05 else 'ns'
        print(f'  {e1} vs {e2}: p_adj={p_adj:.2e} {sig}')

In [ ]:
colors = {'Inflamed': '#E64B35', 'Intermediate': '#4DBBD5', 'Immune-desert': '#91D1C2'}
labels = {'B4GALNT1': 'B4GALNT1\n(GD2 synthase)', 'ST8SIA1': 'ST8SIA1\n(GD3 synthase)', 'IL13RA2': 'IL13RA2', 'ERBB2': 'ERBB2\n(HER2)', 'CD276': 'CD276\n(B7-H3)'}

fig, axes = plt.subplots(1, 5, figsize=(16, 4.5), sharey=False)
fig.suptitle('CAR-T Target Expression Across Immune Ecotypes', fontsize=14, fontweight='bold', y=1.02)

for ax, gene in zip(axes, gene_symbols):
    data_list = [df.loc[df['ecotype']==e, gene].values for e in ecotype_order]
    bp = ax.boxplot(data_list, positions=[0,1,2], widths=0.6, patch_artist=True, showfliers=False, medianprops=dict(color='black', linewidth=1.5))
    for patch, e in zip(bp['boxes'], ecotype_order): patch.set_facecolor(colors[e]); patch.set_alpha(0.7)
    for j, e in enumerate(ecotype_order):
        vals = df.loc[df['ecotype']==e, gene].values
        ax.scatter(j + np.random.uniform(-0.15, 0.15, len(vals)), vals, c=colors[e], s=8, alpha=0.3, edgecolors='none', zorder=3)
    groups = [df.loc[df['ecotype']==e, gene].values for e in ecotype_order]
    H, p_kw = kruskal(*groups)
    p_str = f'p = {p_kw:.1e}' if p_kw < 0.001 else f'p = {p_kw:.3f}' if p_kw < 0.05 else f'p = {p_kw:.2f} (ns)'
    ax.set_title(f'{labels[gene]}\nKW {p_str}', fontsize=9, pad=8)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(['Inflamed','Intermed.','Desert'], fontsize=7.5, rotation=30, ha='right')
    ax.set_ylabel('log2(TPM + 1)', fontsize=9); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig('figs_300dpi/Supplementary_Figure_S_CART_targets.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()